In [2]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob
import math
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import itertools
import random

In [3]:
BASE_DIR = "/projects/standard/kumarv/shared/dwij/daycent/data/SAS_KGML_090925"
INPUT_DIR = os.path.join(BASE_DIR, "InputData")
OUTPUT_DIR = os.path.join(BASE_DIR, "OutputData_Synthetic_10000")
POINTS_LOOKUP = os.path.join(BASE_DIR, "SAS_points_lookup.csv")

PROCESSED_DIR = "/projects/standard/kumarv/shared/dwij/daycent/data/experiment10"

WEATHER_DIR = os.path.join(INPUT_DIR, "WeatherData")
INITC_FN = os.path.join(INPUT_DIR, "initial_site_conditions.xlsx")

MONTHLY_FN = os.path.join(OUTPUT_DIR, "SAS_scenario_1_monthly.csv")
HARVEST_FN = os.path.join(OUTPUT_DIR, "SAS_scenario_1_harvest.csv")

SCENARIOS_FN = os.path.join(INPUT_DIR, "schedule_scenarios_all_Synthetic_10000.csv")

In [18]:
all_points = []

for points in os.listdir(WEATHER_DIR):
    df = pd.read_csv(os.path.join(WEATHER_DIR, points))
    df['point_id'] = points.split(".csv")[0]
    all_points.append(df)

weather_df = pd.concat(all_points, ignore_index=True)
weather_df

,Year,doy,Tmax,Tmin,Precip,point_id
0,2000,1,5.270,-4.830,0.0000,1773513
1,2000,2,12.730,-1.850,0.0000,1773513
2,2000,3,15.020,2.430,0.0520,1773513
3,2000,4,10.480,0.620,1.0260,1773513
4,2000,5,1.870,-5.290,0.0410,1773513
...,...,...,...,...,...,...
1853791,2024,362,12.818,-2.313,0.8766,710977
1853792,2024,363,0.784,-5.354,0.0000,710977
1853793,2024,364,4.451,-5.468,0.0062,710977
1853794,2024,365,3.933,-3.492,0.1565,710977


In [36]:
df = pd.read_csv(POINTS_LOOKUP)

# Calculate medians for splitting
median_x = df['POINT_X'].median()
median_y = df['POINT_Y'].median()

# Create quadrants
df['quadrant'] = 'Q1'
df.loc[(df['POINT_X'] <= median_x) & (df['POINT_Y'] <= median_y), 'quadrant'] = 'Q1 (SW)'
df.loc[(df['POINT_X'] > median_x) & (df['POINT_Y'] <= median_y), 'quadrant'] = 'Q2 (SE)'
df.loc[(df['POINT_X'] <= median_x) & (df['POINT_Y'] > median_y), 'quadrant'] = 'Q3 (NW)'
df.loc[(df['POINT_X'] > median_x) & (df['POINT_Y'] > median_y), 'quadrant'] = 'Q4 (NE)'

train_quadrants = ['Q1 (SW)']
test_quadrants = ['Q2 (SE)', 'Q3 (NW)', 'Q4 (NE)']
train_checker = df[df['quadrant'].isin(train_quadrants)]
test_checker = df[df['quadrant'].isin(test_quadrants)]
print(f"Train (Q1): {len(train_checker)} points")
print(f"Test (Q4): {len(test_checker)} points")

train_pids = train_checker['id'].astype(str).unique()
test_pids = test_checker['id'].astype(str).unique()


Train (Q1): 56 points
Test (Q4): 147 points


In [37]:
train_weather = weather_df[weather_df['point_id'].isin(train_pids)]

#normalise tmax tmin and precip using standard scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_weather[['Tmax', 'Tmin', 'Precip']] = scaler.fit_transform(train_weather[['Tmax', 'Tmin', 'Precip']])
train_weather

,Year,doy,Tmax,Tmin,Precip,point_id
36528,2000,1,-0.885835,-0.952960,-0.393595,1774685
36529,2000,2,-0.245399,-0.633080,-0.393595,1774685
36530,2000,3,-0.089986,-0.262795,-0.351014,1774685
36531,2000,4,-0.402519,-0.420796,1.188763,1774685
36532,2000,5,-1.231670,-1.017905,-0.351014,1774685
...,...,...,...,...,...,...
894931,2024,362,0.046469,-0.636085,0.296901,701081
894932,2024,363,-1.123992,-0.972541,-0.393595,701081
894933,2024,364,-0.897704,-0.924559,-0.319971,701081
894934,2024,365,-0.993343,-0.732146,-0.393595,701081


In [39]:
test_weather = weather_df[~weather_df['point_id'].isin(train_pids)]
test_weather[['Tmax', 'Tmin', 'Precip']] = scaler.transform(test_weather[['Tmax', 'Tmin', 'Precip']])
test_weather

,Year,doy,Tmax,Tmin,Precip,point_id
0,2000,1,-0.898644,-0.958776,-0.393595,1773513
1,2000,2,-0.261623,-0.669915,-0.393595,1773513
2,2000,3,-0.066077,-0.255041,-0.322169,1773513
3,2000,4,-0.453754,-0.430490,1.015693,1773513
4,2000,5,-1.188975,-1.003365,-0.337278,1773513
...,...,...,...,...,...,...
1853791,2024,362,0.094886,-0.599348,0.130561,701977
1853792,2024,363,-1.111951,-0.962653,-0.368596,701977
1853793,2024,364,-0.891556,-0.932216,-0.310906,701977
1853794,2024,365,-0.947060,-0.667298,-0.393595,701977


In [40]:
weather_df = pd.concat([train_weather, test_weather], ignore_index=True)
weather_df

,Year,doy,Tmax,Tmin,Precip,point_id
0,2000,1,-0.885835,-0.952960,-0.393595,1774685
1,2000,2,-0.245399,-0.633080,-0.393595,1774685
2,2000,3,-0.089986,-0.262795,-0.351014,1774685
3,2000,4,-0.402519,-0.420796,1.188763,1774685
4,2000,5,-1.231670,-1.017905,-0.351014,1774685
...,...,...,...,...,...,...
1853791,2024,362,0.094886,-0.599348,0.130561,701977
1853792,2024,363,-1.111951,-0.962653,-0.368596,701977
1853793,2024,364,-0.891556,-0.932216,-0.310906,701977
1853794,2024,365,-0.947060,-0.667298,-0.393595,701977


In [7]:
# vocabulary of management events
MANAGEMENT_CLASSES = [
 'conventional_till_molboadplow','herbicide','soybean_planting',
 'nitrogen_fertilization_1.5gNm2','harvest_grain','cycle_end',
 'ryegrass_planting','nitrogen_fertilization_0gNm2',
 'reduced_till_tandemdisk','notill_rodweederrow','corn_planting',
 'nitrogen_fertilization_17.74gNm2','winterwheat_planting',
 'nitrogen_fertilization_10.312gNm2'
]

scenarios_df = pd.read_csv(SCENARIOS_FN)
scenarios_df = scenarios_df.rename({'simyear': 'Year'},  axis=1)

scenarios_df = scenarios_df.pivot_table(
    index=['scenario', 'Year', 'doy'], # Use all identifying columns for the index
    columns='management',
    aggfunc='size',
    fill_value=0
).reset_index()

scenarios_df = scenarios_df[scenarios_df['scenario'].astype(str).isin([f'scenario_{i}' for i in range(1, 31)])]
scenarios_df

management,scenario,Year,doy,conventional_till_molboadplow,corn_planting,cycle_end,harvest_grain,herbicide,nitrogen_fertilization_0gNm2,nitrogen_fertilization_1.5gNm2,nitrogen_fertilization_10.312gNm2,nitrogen_fertilization_17.74gNm2,notill_rodweederrow,reduced_till_tandemdisk,ryegrass_planting,soybean_planting,winterwheat_planting
0,scenario_1,2000,132,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,scenario_1,2000,136,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,scenario_1,2000,137,0,0,0,0,0,0,1,0,0,0,0,0,1,0
3,scenario_1,2000,289,0,0,1,1,0,0,0,0,0,0,0,0,0,0
4,scenario_1,2000,294,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
965348,scenario_9,2022,289,0,0,1,1,0,0,0,0,0,0,0,0,0,0
965349,scenario_9,2022,294,1,0,0,0,0,0,0,0,0,0,0,0,0,0
965350,scenario_9,2022,315,0,0,0,0,0,0,0,1,0,0,0,0,0,1
965351,scenario_9,2023,200,0,0,1,1,0,0,0,0,0,0,0,0,0,0


In [6]:
scenarios_df.keys()

Index(['scenario', 'Year', 'doy', 'conventional_till_molboadplow',
       'corn_planting', 'cycle_end', 'harvest_grain', 'herbicide',
       'nitrogen_fertilization_0gNm2', 'nitrogen_fertilization_1.5gNm2',
       'nitrogen_fertilization_10.312gNm2', 'nitrogen_fertilization_17.74gNm2',
       'notill_rodweederrow', 'reduced_till_tandemdisk', 'ryegrass_planting',
       'soybean_planting', 'winterwheat_planting'],
      dtype='object', name='management')

In [42]:
numbers = [i for i in range(1, 10001)]
numbers = random.sample(numbers, 50)

print("Selected scenarios:", numbers)

Selected scenarios: [8050, 2765, 6111, 2425, 7695, 2795, 1773, 1803, 9017, 1639, 9109, 1087, 368, 7221, 6305, 8457, 3396, 7727, 5211, 7759, 6563, 9387, 9223, 8180, 5142, 5003, 5533, 9169, 4947, 9319, 2670, 8670, 1273, 5624, 4361, 7737, 4770, 5967, 534, 6909, 6232, 5848, 143, 4283, 7900, 4618, 5089, 9568, 9277, 4006]


In [ ]:
numbers = ['2034', '2600', '1616', '2729', '1000', '3932', '3503', '1390', '4880', '1099']
# numbers = ['1', '10', '100', '1000', '10000', '1001', '2002', '1003', '1004', '1005']
# numbers = [8396, 4951, 9495, 1152, 7888, 3020, 5229, 8900, 6169, 2868]

In [43]:
def load_single_scenario_output(scenario_id: str):
    """Load output data for a single scenario"""
    month_to_doy = {1:30, 2:58, 3:89, 4:119, 5:150, 6:180, 7:211, 8:242, 9:272, 10:303, 11:333, 12:364}
    
    monthly_df = pd.read_csv(os.path.join(OUTPUT_DIR, f"SAS_scenario_{scenario_id}_monthly.csv"))
    monthly_df = monthly_df.rename({'id': 'point_id'}, axis=1)
    monthly_df['doy'] = monthly_df['month'].map(month_to_doy)
    monthly_df['simyear'] = monthly_df['simyear'].apply(lambda x: math.floor(float(x)))

    harvest_df = pd.read_csv(os.path.join(OUTPUT_DIR, f"SAS_scenario_{scenario_id}_harvest.csv"))
    harvest_df = harvest_df.rename({'id': 'point_id', 'dayofyr': 'doy'}, axis=1)

    output_df = pd.merge(monthly_df, harvest_df, on=['runid', 'point_id', 'simyear', 'doy'], how='outer')
    output_df = output_df.rename({'simyear': 'Year'}, axis=1)
    output_df['point_id'] = output_df['point_id'].astype(str)

    dates = pd.to_datetime(output_df['Year'].astype(str) + '-' + output_df['doy'].astype(str), format='%Y-%j')
    output_df['month'].fillna(dates.dt.month, inplace=True)
    output_df['month'] = output_df['month'].astype(int)

    output_df.sort_values(['point_id', 'Year', 'month', 'doy'], inplace=True)
    output_df.sort_index(inplace=True)

    output_df['scenario_id'] = scenario_id
    return output_df


def load_output_data(scenario_ids: list[str], max_workers: int = None):
    """Load output data for multiple scenarios using multithreading"""
    all_outputs = []
    
    # Use ThreadPoolExecutor for I/O-bound operations
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_scenario = {
            executor.submit(load_single_scenario_output, scenario_id): scenario_id 
            for scenario_id in scenario_ids
        }
        
        # Collect results as they complete
        for future in tqdm(as_completed(future_to_scenario), "Output scenarios loaded", total=len(future_to_scenario)):
            scenario_id = future_to_scenario[future]
            try:
                output_df = future.result()
                all_outputs.append(output_df)
            except Exception as exc:
                print(f'Scenario {scenario_id} generated an exception: {exc}')
    
    return pd.concat(all_outputs, ignore_index=True)


def load_management_data(scenario_ids: list[str]):
    scenarios_df = pd.read_csv(SCENARIOS_FN).rename({'simyear': 'Year'}, axis=1)

    scenarios_df = scenarios_df.pivot_table(
        index=['scenario', 'Year', 'doy'],
        columns='management',
        aggfunc='size',
        fill_value=0
    ).reset_index()

    # filter for only requested scenarios
    scenarios_df = scenarios_df[scenarios_df['scenario'].isin([f'scenario_{sid}' for sid in scenario_ids])]

    return scenarios_df


def load_data(scenario_ids: list[str], weather_df: pd.DataFrame, max_workers: int = None):
    print("Loading management data...")
    management_df = load_management_data(scenario_ids)

    # unique sets
    scenarios = management_df["scenario"].unique()
    years = weather_df["Year"].unique()
    doys = weather_df["doy"].unique()

    grid = pd.DataFrame(itertools.product(scenarios, years, doys),
                        columns=["scenario", "Year", "doy"])

    grid_weather = pd.merge(grid, weather_df, on=["Year","doy"], how="left")

    X_daily = pd.merge(grid_weather, management_df, 
                    on=["scenario","Year","doy"], 
                    how="left")
    X_daily.fillna(0, inplace=True)
    # drop doy > 365
    X_daily = X_daily[X_daily['doy'] <= 365]
    X_daily.sort_values(['scenario', 'point_id', 'Year', 'doy'], inplace=True)
    X_daily.reset_index(drop=True, inplace=True)

    print("Loading output data...")
    Y = load_output_data(scenario_ids, max_workers=max_workers)

    return X_daily, Y


# Example usage:
X_daily, Y = load_data(numbers, weather_df, max_workers=10)

Loading management data...
Loading output data...


Output scenarios loaded: 100%|██████████| 50/50 [00:03<00:00, 13.78it/s]


In [44]:
X_daily

,scenario,Year,doy,Tmax,Tmin,Precip,point_id,conventional_till_molboadplow,corn_planting,cycle_end,...,herbicide,nitrogen_fertilization_0gNm2,nitrogen_fertilization_1.5gNm2,nitrogen_fertilization_10.312gNm2,nitrogen_fertilization_17.74gNm2,notill_rodweederrow,reduced_till_tandemdisk,ryegrass_planting,soybean_planting,winterwheat_planting
0,scenario_1087,2000,1,-0.898644,-0.958776,-0.393595,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,scenario_1087,2000,2,-0.261623,-0.669915,-0.393595,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,scenario_1087,2000,3,-0.066077,-0.255041,-0.322169,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,scenario_1087,2000,4,-0.453754,-0.430490,1.015693,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,scenario_1087,2000,5,-1.188975,-1.003365,-0.337278,1773513,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92618745,scenario_9568,2024,361,-0.772947,-0.531882,-0.329037,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92618746,scenario_9568,2024,362,-0.254109,-0.714795,0.810481,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92618747,scenario_9568,2024,363,-1.281710,-1.009569,-0.393595,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92618748,scenario_9568,2024,364,-0.968579,-1.020620,-0.385079,710977,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
Y['scenario'] = Y['scenario_id'].apply(lambda x: f'scenario_{x}')

import joblib
# read the scaler_Y
# scaler_Y = joblib.load(os.path.join(PROCESSED_DIR, "scaler_Y.pkl"))
scaler_Y = StandardScaler()
train_Y = Y[Y['point_id'].isin(train_pids)]
train_Y[['somsc', 'cgrain']] = scaler_Y.fit_transform(train_Y[['somsc', 'cgrain']])
test_Y = Y[Y['point_id'].isin(test_pids)]
test_Y[['somsc', 'cgrain']] = scaler_Y.transform(test_Y[['somsc', 'cgrain']])

Y = pd.concat([train_Y, test_Y])
Y.sort_values(['scenario', 'point_id', 'Year', 'doy'], inplace=True)
Y

,runid,point_id,Year,month,somsc,doy,cgrain,scenario_id,scenario
638379,104,1773513,2000,10,NaN,289,-0.747877,1087,scenario_1087
638380,104,1773513,2001,1,-1.858963,30,NaN,1087,scenario_1087
638381,104,1773513,2001,2,-1.857565,58,NaN,1087,scenario_1087
638382,104,1773513,2001,3,-1.856493,89,NaN,1087,scenario_1087
638383,104,1773513,2001,4,-1.855698,119,NaN,1087,scenario_1087
...,...,...,...,...,...,...,...,...,...
3007578,103,710977,2023,10,NaN,289,-0.601936,9568,scenario_9568
3007579,103,710977,2023,10,-1.976667,303,NaN,9568,scenario_9568
3007580,103,710977,2023,11,-1.967401,333,NaN,9568,scenario_9568
3007581,103,710977,2023,12,-1.963365,364,NaN,9568,scenario_9568


In [46]:
Y['somsc'].describe(), Y['cgrain'].describe()

(count    2.811550e+06
 mean    -5.173124e-01
 std      1.422439e+00
 min     -5.445342e+00
 25%     -1.392787e+00
 50%     -1.484942e-01
 75%      5.665115e-01
 max      3.352507e+00
 Name: somsc, dtype: float64,
 count    225533.000000
 mean         -0.087360
 std           0.972892
 min          -1.858741
 25%          -0.793130
 50%          -0.565928
 75%           0.958149
 max           2.326683
 Name: cgrain, dtype: float64)

In [47]:
# save scaler_Y
import joblib
joblib.dump(scaler_Y, os.path.join(PROCESSED_DIR, "scaler_Y_50.pkl"))

['/projects/standard/kumarv/shared/dwij/daycent/data/experiment10/scaler_Y_50.pkl']

# Preprocessing

## Inputs processing

In [48]:
def create_and_save_sequences(X_daily: pd.DataFrame, Y: pd.DataFrame, pids: list[str], file_prefix: str):
    """Create sequences and save to .npy files"""
    df = X_daily[X_daily['point_id'].isin(pids)]

    # Step 2: Select feature columns (include doy, exclude Year & point_id)
    feature_cols = [c for c in df.columns if c not in ["scenario", "Year", "point_id"]]

    # Step 3: Group by point_id and Year
    groups = df.groupby(["scenario", "Year", "point_id"])

    # Step 4: Create sequences and store mapping
    sequences = []
    mapping = []  # to store (scenario, point_id, year) for each sequence

    for (sid, pid, year), group in tqdm(groups):
        sequences.append(group[feature_cols].to_numpy())
        mapping.append((sid, pid, year))  # store mapping info

    # Convert to arrays
    sequences = np.stack(sequences)  # shape: (num_sequences, 365, num_features)
    mapping = np.array(mapping)      # shape: (num_sequences, 2)

    # Step 5: Save both sequences and mapping
    # Step 5: Save everything into one npy file
    data_dict = {
        "data": sequences,
        "mapping": mapping,
        "columns": feature_cols
    }

    np.save(os.path.join(PROCESSED_DIR, f"{file_prefix}_X.npy"), data_dict, allow_pickle=True)

    Y_temp = Y[Y['point_id'].isin(pids)]
    # Build dictionary keyed by (point_id, Year)
    Y_dict = {}
    for (sid, pid, year), group in tqdm(Y_temp.groupby(["scenario", "point_id", "Year"])):
        Y_dict[(sid, pid, year)] = group[["month", "doy", "somsc", "cgrain"]].to_numpy()
    # Align Y to mapping
    somsc_list = []
    cgrain_list = []
    for sid, year, pid in tqdm(mapping, desc="Aligning Y to mapping"):
        if (str(sid), str(pid), int(year)) in Y_dict:
            data = Y_dict[str(sid), str(pid), int(year)]

            somsc_array = np.full(12, np.nan, dtype=np.float64)
            
            # The 'data' array has columns: 0=month, 1=doy, 2=somsc, 3=cgrain
            
            # 1a. Create a boolean mask to filter rows where 'somsc' (column index 2) is NOT NaN
            valid_somsc_mask = ~np.isnan(data[:, 2])
            
            # 1b. Filter the data to include only rows with valid somsc values
            valid_data = data[valid_somsc_mask]
            
            # 1c. Get the 0-indexed positions for assignment: month (column 0) - 1
            # We must ensure the indices are integers
            indices = valid_data[:, 0].astype(int) - 1
            
            # 1d. Get the corresponding somsc values (column 2)
            values = valid_data[:, 2]
            
            # 1e. Use advanced NumPy indexing for vectorized assignment
            # This is much faster than iterating row by row.
            # Note: If there are multiple somsc values for the same month, 
            # the last value encountered in the 'data' array (due to sorting in Y_dict) will be used.
            if indices.size > 0:
                somsc_array[indices] = values

                # 2. cgrain value: Must be a single number (no NaNs allowed in the source data)
            # Extract all cgrain values for this year (column index 3)
            cgrain_values = data[:, 3]
            
            # Filter out NaN values to find the single valid cgrain number
            valid_cgrain = cgrain_values[~np.isnan(cgrain_values)]
            
            # Append the results
            if valid_cgrain.size > 0:
                # Append the single annual cgrain value (the user guarantees it's unique/present)
                cgrain_list.append(valid_cgrain[0]) 
                
                # Append the 12-element monthly somsc array
                somsc_list.append(somsc_array)
            else:
                # Handle the case where Cgrain is unexpectedly missing (use NaN as a fallback)
                # print(f"Warning: cgrain value is missing for point_id {pid}, Year {year}. Appending NaN.")
                somsc_list.append(somsc_array)
                cgrain_list.append(np.nan)
            

        else:
            print(f"Missing data for point_id {pid}, Year {year}, scenario {sid}. ")
            somsc_list.append(np.full(12, np.nan, dtype=np.float64))
            cgrain_list.append(np.nan)


    # Convert lists to final NumPy arrays
    final_somsc_array = np.array(somsc_list)
    final_cgrain_array = np.array(cgrain_list)

    # --- RESULTS ---
    print("\n--- Final Results ---")
    print("Mapping length:", len(mapping))
    print("SOMSC List length:", len(somsc_list))
    print("CGRAIN List length:", len(cgrain_list))

    print(f"\nFinal SOMSC Array (Shape: {final_somsc_array.shape}):")
    print(final_somsc_array)

    print(f"\nFinal CGRAIN Array (Shape: {final_cgrain_array.shape}):")
    print(final_cgrain_array)

    data_dict = {
        "somsc": final_somsc_array,
        "cgrain": final_cgrain_array,
    }

    np.save(os.path.join(PROCESSED_DIR, f"{file_prefix}_Y.npy"), data_dict, allow_pickle=True)



In [50]:
# create_and_save_sequences(X_daily, Y, train_pids, "train_50")
create_and_save_sequences(X_daily, Y, test_pids, "test_50")

Aligning Y to mapping:  12%|█▏        | 21639/183750 [00:00<00:02, 72723.55it/s]

Missing data for point_id 1773513, Year 2000, scenario scenario_1639. 
Missing data for point_id 1773576, Year 2000, scenario scenario_1639. 
Missing data for point_id 1773749, Year 2000, scenario scenario_1639. 
Missing data for point_id 1773883, Year 2000, scenario scenario_1639. 
Missing data for point_id 1774020, Year 2000, scenario scenario_1639. 
Missing data for point_id 1774198, Year 2000, scenario scenario_1639. 
Missing data for point_id 1774528, Year 2000, scenario scenario_1639. 
Missing data for point_id 1774539, Year 2000, scenario scenario_1639. 
Missing data for point_id 1774853, Year 2000, scenario scenario_1639. 
Missing data for point_id 1775042, Year 2000, scenario scenario_1639. 
Missing data for point_id 1775693, Year 2000, scenario scenario_1639. 
Missing data for point_id 1776301, Year 2000, scenario scenario_1639. 
Missing data for point_id 1776510, Year 2000, scenario scenario_1639. 
Missing data for point_id 1776558, Year 2000, scenario scenario_1639. 
Missin

Aligning Y to mapping:  24%|██▍       | 43929/183750 [00:00<00:01, 74037.11it/s]

Missing data for point_id 1773513, Year 2000, scenario scenario_2795. 
Missing data for point_id 1773576, Year 2000, scenario scenario_2795. 
Missing data for point_id 1773749, Year 2000, scenario scenario_2795. 
Missing data for point_id 1773883, Year 2000, scenario scenario_2795. 
Missing data for point_id 1774020, Year 2000, scenario scenario_2795. 
Missing data for point_id 1774198, Year 2000, scenario scenario_2795. 
Missing data for point_id 1774528, Year 2000, scenario scenario_2795. 
Missing data for point_id 1774539, Year 2000, scenario scenario_2795. 
Missing data for point_id 1774853, Year 2000, scenario scenario_2795. 
Missing data for point_id 1775042, Year 2000, scenario scenario_2795. 
Missing data for point_id 1775693, Year 2000, scenario scenario_2795. 
Missing data for point_id 1776301, Year 2000, scenario scenario_2795. 
Missing data for point_id 1776510, Year 2000, scenario scenario_2795. 
Missing data for point_id 1776558, Year 2000, scenario scenario_2795. 
Missin

Aligning Y to mapping:  44%|████▍     | 81529/183750 [00:01<00:01, 75100.68it/s]

Missing data for point_id 1773513, Year 2000, scenario scenario_5089. 
Missing data for point_id 1773576, Year 2000, scenario scenario_5089. 
Missing data for point_id 1773749, Year 2000, scenario scenario_5089. 
Missing data for point_id 1773883, Year 2000, scenario scenario_5089. 
Missing data for point_id 1774020, Year 2000, scenario scenario_5089. 
Missing data for point_id 1774198, Year 2000, scenario scenario_5089. 
Missing data for point_id 1774528, Year 2000, scenario scenario_5089. 
Missing data for point_id 1774539, Year 2000, scenario scenario_5089. 
Missing data for point_id 1774853, Year 2000, scenario scenario_5089. 
Missing data for point_id 1775042, Year 2000, scenario scenario_5089. 
Missing data for point_id 1775693, Year 2000, scenario scenario_5089. 
Missing data for point_id 1776301, Year 2000, scenario scenario_5089. 
Missing data for point_id 1776510, Year 2000, scenario scenario_5089. 
Missing data for point_id 1776558, Year 2000, scenario scenario_5089. 
Missin

Aligning Y to mapping:  86%|████████▌ | 157135/183750 [00:02<00:00, 75643.38it/s]

Missing data for point_id 1773513, Year 2000, scenario scenario_8180. 
Missing data for point_id 1773576, Year 2000, scenario scenario_8180. 
Missing data for point_id 1773749, Year 2000, scenario scenario_8180. 
Missing data for point_id 1773883, Year 2000, scenario scenario_8180. 
Missing data for point_id 1774020, Year 2000, scenario scenario_8180. 
Missing data for point_id 1774198, Year 2000, scenario scenario_8180. 
Missing data for point_id 1774528, Year 2000, scenario scenario_8180. 
Missing data for point_id 1774539, Year 2000, scenario scenario_8180. 
Missing data for point_id 1774853, Year 2000, scenario scenario_8180. 
Missing data for point_id 1775042, Year 2000, scenario scenario_8180. 
Missing data for point_id 1775693, Year 2000, scenario scenario_8180. 
Missing data for point_id 1776301, Year 2000, scenario scenario_8180. 
Missing data for point_id 1776510, Year 2000, scenario scenario_8180. 
Missing data for point_id 1776558, Year 2000, scenario scenario_8180. 
Missin

Aligning Y to mapping: 100%|██████████| 183750/183750 [00:02<00:00, 74782.95it/s]



--- Final Results ---
Mapping length: 183750
SOMSC List length: 183750
CGRAIN List length: 183750

Final SOMSC Array (Shape: (183750, 12)):
[[        nan         nan         nan ...         nan         nan
          nan]
 [        nan         nan         nan ...         nan         nan
          nan]
 [        nan         nan         nan ...         nan         nan
          nan]
 ...
 [-3.75633139         nan         nan ...         nan         nan
          nan]
 [-1.27793887         nan         nan ...         nan         nan
          nan]
 [-1.96091916         nan         nan ...         nan         nan
          nan]]

Final CGRAIN Array (Shape: (183750,)):
[-0.74787726 -0.73580207 -0.67393158 ...         nan         nan
         nan]
